In [9]:
import pandas as pd
import numpy as np

import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from pytorch_tabnet.tab_model import TabNetClassifier

# Modellenmeye hazır veri setinin okunması
df = pd.read_csv("preprocessed_common.csv")


In [10]:
# Kategorik değişkenlerin Label Encoding ile sayısallaştırılması
# (TabNet modelinde kategorik değişkenlerin one-hot encoding ile parçalanması 
# uygun olmayacağı için label encoding yöntemi kullanıldı)
categorical_cols = df.select_dtypes(include=['object','category','bool']).columns

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print("✔ TabNet için Label Encoding tamamlandı.")


✔ TabNet için Label Encoding tamamlandı.


In [11]:
# Hedef değişken
target = "is_onview_arrest"

# Özellikler (X) ve hedef (y)
X = df.drop(columns=[target]).values
y = df[target].values


In [12]:
# Train/Test Bölme
# Sınıf dengesini korumak için stratify=y kullandık
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [13]:
# TabNet modelinin tanımlanması
tabnet = TabNetClassifier(
    n_d=32,
    n_a=32,
    n_steps=5,
    gamma=1.5,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=0.02),
    mask_type="entmax",
    seed=42
)


C:\Users\miray\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [14]:
# Model eğitimi
tabnet.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric=["auc"],
    max_epochs=100,
    patience=15,
    batch_size=1024,
    virtual_batch_size=128,
    num_workers=0
)


epoch 0  | loss: 0.60563 | val_0_auc: 0.56917 |  0:00:08s
epoch 1  | loss: 0.52977 | val_0_auc: 0.5     |  0:00:15s
epoch 2  | loss: 0.51495 | val_0_auc: 0.56671 |  0:00:23s
epoch 3  | loss: 0.50756 | val_0_auc: 0.60344 |  0:00:30s
epoch 4  | loss: 0.50678 | val_0_auc: 0.6099  |  0:00:37s
epoch 5  | loss: 0.50464 | val_0_auc: 0.57516 |  0:00:43s
epoch 6  | loss: 0.50413 | val_0_auc: 0.64539 |  0:00:52s
epoch 7  | loss: 0.50338 | val_0_auc: 0.58852 |  0:00:59s
epoch 8  | loss: 0.5026  | val_0_auc: 0.61061 |  0:01:07s
epoch 9  | loss: 0.50435 | val_0_auc: 0.60374 |  0:01:14s
epoch 10 | loss: 0.50054 | val_0_auc: 0.5589  |  0:01:22s
epoch 11 | loss: 0.49802 | val_0_auc: 0.60214 |  0:01:29s
epoch 12 | loss: 0.49788 | val_0_auc: 0.63942 |  0:01:38s
epoch 13 | loss: 0.4952  | val_0_auc: 0.70119 |  0:01:45s
epoch 14 | loss: 0.49155 | val_0_auc: 0.5182  |  0:01:52s
epoch 15 | loss: 0.49236 | val_0_auc: 0.71102 |  0:01:59s
epoch 16 | loss: 0.49014 | val_0_auc: 0.71323 |  0:02:06s
epoch 17 | los

C:\Users\miray\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [16]:
from sklearn.metrics import classification_report, accuracy_score, f1_score, recall_score, precision_score, roc_auc_score

# Tahminler
y_pred = tabnet.predict(X_test)
y_prob = tabnet.predict_proba(X_test)[:,1]

# Model değerlendirmesi
print("=== TabNet Classifier ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("-" * 30)
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred, digits=4))

=== TabNet Classifier ===
Accuracy: 0.8424491516492781
Precision: 0.7186982901268616
Recall: 0.569493006993007
F1: 0.6354547671299683
ROC-AUC: 0.8766746897041301
------------------------------
Sınıflandırma Raporu:
              precision    recall  f1-score   support

           0     0.8717    0.9292    0.8995      7201
           1     0.7187    0.5695    0.6355      2288

    accuracy                         0.8424      9489
   macro avg     0.7952    0.7493    0.7675      9489
weighted avg     0.8348    0.8424    0.8358      9489

